In [1]:
import sys

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.graph_stats as graph_stats
import utils.graph_utils as graph_utils
import utils.vis as vis

from config_handlers.sk_handler import SKHandler

# Main

In [2]:
sk_hdl = SKHandler()
DATASETS = sk_hdl.get_ds_list()
base_dir = Path("..")

all_datasets = DATASETS * 3
all_datasets.sort()

In [3]:
# Loading concept and predicate frequencies from VG1800 dataset.
# VG1800 is a subsample of VG made by the authors of IETrans.
vg1800_dict = graph_utils.load_vg1800_dict()

In [4]:
def get_sk_data() -> dict:
    all_sks = {}

    for ds in DATASETS:
        sk_hdl.set_curr_ds(ds)
        if ds not in all_sks:
            all_sks[ds] = {}

        for ds_class in sk_hdl.get_classes():
            sk_hdl.set_curr_class(ds_class)
            sk_path = base_dir.joinpath(sk_hdl.get_sk_final_path())
            sk_data = data_io.load_jsonl(sk_path)

            if ds_class not in all_sks[ds]:
                all_sks[ds][ds_class] = {}

            sk_dict = {}
            for curr_sample in sk_data:
                question_id = curr_sample["question_id"]
                if question_id not in sk_dict:
                    sk_dict[question_id] = curr_sample["relations"]
            all_sks[ds][ds_class] = sk_dict

    return all_sks

In [5]:
def get_sk_details(sks: dict) -> dict:
    data = {
        "graphs": {ds: {} for ds in DATASETS},
        "summaries": {ds: {} for ds in DATASETS},
        "sample_stats": {ds: {} for ds in DATASETS},
        "count_concepts": {ds: {} for ds in DATASETS},
        "count_preds": {ds: {} for ds in DATASETS},
    }

    for ds in sks.keys():
        print(f"Processing {ds}")
        sk_hdl.set_curr_ds(ds)

        for ds_class in sks[ds].keys():
            sk_hdl.set_curr_class(ds_class)

            # No distinction between classes, put everything from 1 dataset together
            for sk_idx, sk_data in sks[ds][ds_class].items():
                data["graphs"][ds][sk_idx] = graph_utils.sk_to_nx(sk_data)

            sample_stats, stats_summary = graph_stats.compute_stats(
                data["graphs"][ds], vg1800_dict
            )

        # Update
        data["sample_stats"][ds] = sample_stats
        data["summaries"][ds] = stats_summary
        data["count_concepts"][ds] = stats_summary["count_concepts"]
        data["count_preds"][ds] = stats_summary["count_preds"]

    return data

In [6]:
# Load SK data
sks = get_sk_data()
# Process SK data
sk_data = get_sk_details(sks)

Processing llava-bench
Processing mmbench
Processing seed
Processing vqav2


In [7]:
# for k, v in sk_data["summaries"]["mmbench"].items():
#     print(f"{k.upper()}: {v:.2f} -- {type(v)}")

# Save results to disk

Save statistics

In [7]:
out_dir = Path(".", "0_stats", "sk")
data_io.make_dir(out_dir)

for ds in DATASETS:
    print(f"Saving {ds}")

    # Summaries
    summary_file = out_dir.joinpath(ds, "summary.json")
    data_io.make_dir(summary_file.parent)
    data_io.save_json(sk_data["summaries"][ds], summary_file)
    # Sample-level
    sample_file = out_dir.joinpath(ds, "samples.json")
    data_io.save_json(sk_data["sample_stats"][ds], sample_file)

[Errno 17] File exists: '0_stats/sk'
Saving llava-bench
[Errno 17] File exists: '0_stats/sk/llava-bench'
Saving mmbench
[Errno 17] File exists: '0_stats/sk/mmbench'
Saving seed
[Errno 17] File exists: '0_stats/sk/seed'
Saving vqav2
[Errno 17] File exists: '0_stats/sk/vqav2'


Save statistics graphs

In [8]:
# Create output dir for charts
charts_dir = Path(".", "stats_charts")
data_io.make_dir(charts_dir)

[Errno 17] File exists: 'stats_charts'


In [9]:
all_count_concepts = {}
all_count_preds = {}

for ds in DATASETS:
    sk_hdl.set_curr_ds(ds)
    all_count_concepts[ds] = {"init": [], "val": [], "ann": []}
    all_count_preds[ds] = {"init": [], "val": [], "ann": []}
    for ds_class in sk_hdl.get_classes():
        sk_hdl.set_curr_class(ds_class)

        # Loading and formatting stats
        stats_path = sk_hdl.get_sk_final_path().parent.joinpath("stats.json")
        stats_path = base_dir.joinpath(stats_path)
        stats = data_io.load_json(stats_path)
        for k, v in stats["samples"].items():
            # Concepts
            all_count_concepts[ds]["init"].append(v["concepts"]["init"])
            all_count_concepts[ds]["val"].append(v["concepts"]["val"])
            all_count_concepts[ds]["ann"].append(v["concepts"]["ann"])
            # Relations
            all_count_preds[ds]["init"].append(v["preds"]["init"])
            all_count_preds[ds]["val"].append(v["preds"]["val"])
            all_count_preds[ds]["ann"].append(v["preds"]["ann"])

In [10]:
# Doing one with only the relations
vis.plot_relations(all_count_preds, charts_dir.joinpath("box_sk.pdf"))

['llava-bench', 'mmbench', 'seed', 'vqav2']


/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:222: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:222: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:222: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:222: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])


In [11]:
# Doing one with both relations and concepts
vis.plot_sk(all_count_concepts, all_count_preds, charts_dir.joinpath("box_sk_full.pdf"))

/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:170: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", "", "Relations", ""])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:170: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", "", "Relations", ""])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:170: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", "", "Relations", ""])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:170: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", ""

# Old

To save as Excel files

In [ ]:
# # Save to Excel with two sheets
# sample_stats_df = pd.DataFrame(sample_stats)
# dataset_stats_df = pd.DataFrame([dataset_stats])
# output_path = f"output/{step}/graph_statistics.xlsx"
# with pd.ExcelWriter(output_path) as writer:
#     sample_stats_df.to_excel(writer, sheet_name="Sample Statistics", index=False)
#     dataset_stats_df.to_excel(writer, sheet_name="Dataset Summary", index=False)